# 1단계 실험: 주제 → 검색 → 요약 (Research Chain)

`src/llm_jacky/research.py` 의 동작을 단계별로 분해해 실험합니다.

**준비**: 프로젝트 루트의 `.env` 에 `OPENAI_API_KEY`, `TAVILY_API_KEY` 가 들어 있어야 합니다.

## 0. 환경 설정

In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

for k in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    print(f"{k}: {'OK' if os.getenv(k) else 'MISSING'}")

OPENAI_API_KEY: OK
TAVILY_API_KEY: OK


## 1. End-to-end 실행

주제를 바꿔가며 전체 파이프라인을 한 번에 호출해 봅니다.

In [2]:
from llm_jacky.research import run_research

TOPIC = "외국인을 위한 한국 길거리 음식 추천"

result = run_research(TOPIC, max_results=5)
print("=== 요약 ===")
print(result.summary)
print("\n=== 출처 ===")
for i, s in enumerate(result.sources, 1):
    print(f"[{i}] {s.title}\n    {s.url}")

/Users/jackykim/project/llm-jacky/src/llm_jacky/research.py:66: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults(max_results=max_results)


=== 요약 ===
1) 한국의 길거리 음식은 다양하고 맛있으며, 외국인들에게도 큰 인기를 끌고 있다. 대표적인 메뉴로는 떡볶이, 김밥, 핫도그, 호떡 등이 있으며, 이들은 한국의 길거리 문화와 함께 즐길 수 있는 음식들이다. 외국인들은 이러한 음식들을 통해 한국의 독특한 맛과 문화를 경험할 수 있다.

2) 주요 사실/포인트:
- **떡볶이**: 매콤하고 달콤한 소스에 쫀득한 떡과 어묵이 어우러진 한국의 대표 길거리 음식.
- **김밥**: 다양한 재료가 들어간 건강한 간식으로, 소풍이나 간단한 한 끼로 인기가 높음.
- **핫도그**: 바삭한 겉과 촉촉한 속이 특징인 한국 전통 핫도그, 다양한 토핑이 가능.
- **호떡**: 겨울철 인기 간식으로, 달콤한 속이 들어간 팬케이크 형태의 음식.
- **길거리 음식 문화**: 한국의 길거리 음식은 저렴하고 맛있으며, 다양한 종류가 있어 여러 사람과 나눠 먹기 좋음.

3) 출처별 한 줄 요약:
- [2] 꼭 먹어봐야 할 한국 길거리 음식 10가지 - https://loveukorea.com/ko/top-10-korean-street-foods-you-must-try/
- [3] 외국인 남편과 한국여행 - https://m.blog.naver.com/wjwj38/223176755965

=== 출처 ===
[1] 세계 길거리 음식 BEST 10 — 한국인이 꼭 먹어봐야 할 이색 메뉴 : 국제 : 기독일보
    https://www.christiandaily.co.kr/news/159366
[2] Top 10 Korean Street Foods You Must Try - Love U Korea
    https://loveukorea.com/ko/top-10-korean-street-foods-you-must-try/
[3] [외국인 남편과 한국여행] 외국인이 좋아하는 한국음식 추천! 꼭 먹어봐야 하는 한식들 놓치지 마세요~ : 네이버 블로그
    https://m.blog.naver.com/wjwj38/2231

## 2. 검색 단계만 보기 (Tavily raw 결과)

어떤 자료가 들어오는지 직접 확인합니다. `max_results`, `search_depth` 등을 바꿔 실험해 보세요.

In [3]:
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults(max_results=5)
raw = search.invoke(TOPIC)

print(f"hits: {len(raw)}\n")
for i, r in enumerate(raw, 1):
    print(f"[{i}] {r.get('title')}")
    print(f"    URL: {r.get('url')}")
    snippet = (r.get('content') or '')[:160].replace('\n', ' ')
    print(f"    snippet: {snippet}...\n")

hits: 5

[1] 세계 길거리 음식 BEST 10 — 한국인이 꼭 먹어봐야 할 이색 메뉴 : 국제 : 기독일보
    URL: https://www.christiandaily.co.kr/news/159366
    snippet: ### 1. 태국 — 팟타이와 망고 스티키 라이스  방콕의 골목 어디에서나 “촹촹” 하는 웍 소리가 들리면, 그 옆 포장마차에는 어김없이 팟타이 줄이 서 있다. 쌀국수면을 강한 불에 빠르게 볶고, 새우, 두부, 부추, 숙주, 라임, 땅콩가루를 마지막에 얹어주는 조합은 그 자체로 “세계인...

[2] Top 10 Korean Street Foods You Must Try - Love U Korea
    URL: https://loveukorea.com/ko/top-10-korean-street-foods-you-must-try/
    snippet: Love U Korea  Top 10 Korean Street Foods You Must Try  # 꼭 먹어봐야 할 한국 길거리 음식 10가지  한국에는 지역, 도시마다 길거리에 다양한 음식이 있습니다. 물론 모든 음식이 맛있겠지만 이중에서 가장 맛있고 한국인에게 인기있는 음식종류와 ...

[3] [외국인 남편과 한국여행] 외국인이 좋아하는 한국음식 추천! 꼭 먹어봐야 하는 한식들 놓치지 마세요~ : 네이버 블로그
    URL: https://m.blog.naver.com/wjwj38/223176755965
    snippet: ​  교촌 허니콤보  ​  솔직히 한국 치킨은 두말하면 입아픈데  ​  치킨중에서도 교촌 허니콤보 강추합니다!  ​  ​  한국 치킨 거기서 거기지  많이 먹어봤어~  하는 외국인한테도 먹히는 맛  ​  ​  한국 치킨 거기서 거기라고? 누가 그러니~?  ​  허니콤보 들어간다 입벌려 ...

[4] 외국인이 좋아하는 한식 49선 - 램프쿡
    URL: https://www.lampcook.com/cook/food_top100_list1.php

## 3. 프롬프트 확인

LLM 에 실제로 들어가는 프롬프트 텍스트를 점검합니다. 프롬프트 튜닝 시작점.

In [4]:
from llm_jacky.research import SUMMARY_PROMPT, _format_results

formatted = _format_results(raw)
messages = SUMMARY_PROMPT.format_messages(topic=TOPIC, results=formatted)
for m in messages:
    print(f"--- {m.type.upper()} ---")
    print(m.content[:1200])
    print()

--- SYSTEM ---
당신은 블로그 작성용 리서치 어시스턴트입니다. 주어진 검색 결과만을 근거로 주제에 대한 핵심 사실과 인용 가능한 포인트를 한국어로 정리합니다. 검색 결과에 없는 정보는 절대 만들지 않습니다.

--- HUMAN ---
주제: 외국인을 위한 한국 길거리 음식 추천

검색 결과:
[1] 세계 길거리 음식 BEST 10 — 한국인이 꼭 먹어봐야 할 이색 메뉴 : 국제 : 기독일보
URL: https://www.christiandaily.co.kr/news/159366
### 1. 태국 — 팟타이와 망고 스티키 라이스

방콕의 골목 어디에서나 “촹촹” 하는 웍 소리가 들리면, 그 옆 포장마차에는 어김없이 팟타이 줄이 서 있다. 쌀국수면을 강한 불에 빠르게 볶고, 새우, 두부, 부추, 숙주, 라임, 땅콩가루를 마지막에 얹어주는 조합은 그 자체로 “세계인의 입맛 표준”이다. 매콤·새콤·달콤의 균형이 한국인 입맛에도 잘 맞아 “태국 입문 메뉴”로 늘 추천된다. 망고 스티키 라이스는 잘 익은 망고와 코코넛 밀크에 적신 찹쌀밥의 조합으로, 디저트로 손색이 없다. 가격대는 한국 분식 수준이거나 더 저렴하지만, 풍성함은 어마어마하다. 야시장과 차이나타운의 골목길에서 가장 진하게 즐길 수 있다.

### 2. 베트남 — 반미와 쌀국수

베트남 길거리의 상징은 단연 “반미”다. 프랑스식 바게트에 베트남식 발효 야채, 햄, 고기, 고수, 고추를 얹은 이 샌드위치는 동서양의 절묘한 만남이다. 한 손에 들고 먹기에 완벽한 사이즈와 합리적 가격이 매력이다. 호치민과 하노이의 노점은 이른 아침부터 줄이 길게 늘어선다. 쌀국수 “포”는 또 다른 길거리 대표 메뉴인데, 길거리 노점에서 받아 든 한 그릇이 호텔 다이닝 코스 요리만큼이나 깊이가 있다. 한국인 여행객이 비교적 수월하게 첫 도전을 할 수 있는 길거리 메뉴라는 점이 큰 강점이다. 추가 메뉴로는 분짜, 짜조, 짜요까지 라인업이 풍성하다.

### 3. 멕시코 — 타코 알 파스토르 [...] ### 8. 인도 — 

## 4. 요약 단계만 다시 호출 (프롬프트/모델 변경 실험)

검색은 한 번만 하고, 요약 프롬프트나 모델만 바꿔 비교합니다.

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

custom_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 영어권 독자에게 한국 문화를 소개하는 블로그의 리서처입니다. "
                "검색 결과만 근거로, 인용 가능한 사실 위주로 정리하세요."),
    ("human", "주제: {topic}\n\n검색 결과:\n{results}\n\n"
              "한국어 요약 5문장 + 영어 키워드 5개를 출력하세요."),
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = custom_prompt | llm | StrOutputParser()
print(chain.invoke({"topic": TOPIC, "results": formatted}))

### 한국 길거리 음식 추천 요약

1. **떡볶이**: 매콤하고 달콤한 소스에 쫀득한 떡과 어묵이 어우러진 대표적인 한국 길거리 음식으로, 특히 10대와 20대 여성들에게 인기가 많다.
2. **김밥**: 일본의 스시와 유사한 한국의 김밥은 밥과 다양한 재료가 들어간 건강한 간식으로, 소풍이나 간단한 한 끼로 자주 소비된다.
3. **핫도그**: 바삭한 겉과 촉촉한 속이 특징인 한국식 핫도그는 다양한 토핑과 소스를 곁들여 즐길 수 있다.
4. **호떡**: 겨울철 인기 간식으로, 달콤한 설탕과 견과류가 들어간 한국식 팬케이크이다.
5. **닭꼬치**: 숯불에 구운 닭고기를 양념과 함께 즐길 수 있는 간편한 길거리 음식으로, 한국의 프라이드 치킨과 유사한 맛을 제공한다.

### Keywords
1. Tteokbokki
2. Gimbap
3. Hotdog
4. Hoddeok
5. Dakkochi


## 5. 결과 객체 직렬화 (다음 단계로 넘기기 위해)

2단계(RAG 저장)에서 이 결과를 입력으로 받을 예정이라, 형태를 미리 확인해 둡니다.

In [6]:
from dataclasses import asdict
import json

payload = {
    "topic": result.topic,
    "summary": result.summary,
    "sources": [asdict(s) for s in result.sources],
}
print(json.dumps(payload, ensure_ascii=False, indent=2)[:1500])

{
  "topic": "외국인을 위한 한국 길거리 음식 추천",
  "summary": "1) 한국의 길거리 음식은 다양하고 맛있으며, 외국인들에게도 큰 인기를 끌고 있다. 대표적인 메뉴로는 떡볶이, 김밥, 핫도그, 호떡 등이 있으며, 이들은 한국의 길거리 문화와 함께 즐길 수 있는 음식들이다. 외국인들은 이러한 음식들을 통해 한국의 독특한 맛과 문화를 경험할 수 있다.\n\n2) 주요 사실/포인트:\n- **떡볶이**: 매콤하고 달콤한 소스에 쫀득한 떡과 어묵이 어우러진 한국의 대표 길거리 음식.\n- **김밥**: 다양한 재료가 들어간 건강한 간식으로, 소풍이나 간단한 한 끼로 인기가 높음.\n- **핫도그**: 바삭한 겉과 촉촉한 속이 특징인 한국 전통 핫도그, 다양한 토핑이 가능.\n- **호떡**: 겨울철 인기 간식으로, 달콤한 속이 들어간 팬케이크 형태의 음식.\n- **길거리 음식 문화**: 한국의 길거리 음식은 저렴하고 맛있으며, 다양한 종류가 있어 여러 사람과 나눠 먹기 좋음.\n\n3) 출처별 한 줄 요약:\n- [2] 꼭 먹어봐야 할 한국 길거리 음식 10가지 - https://loveukorea.com/ko/top-10-korean-street-foods-you-must-try/\n- [3] 외국인 남편과 한국여행 - https://m.blog.naver.com/wjwj38/223176755965",
  "sources": [
    {
      "title": "세계 길거리 음식 BEST 10 — 한국인이 꼭 먹어봐야 할 이색 메뉴 : 국제 : 기독일보",
      "url": "https://www.christiandaily.co.kr/news/159366",
      "content": "### 1. 태국 — 팟타이와 망고 스티키 라이스\n\n방콕의 골목 어디에서나 “촹촹” 하는 웍 소리가 들리면, 그 옆 포장마차에는 어김없이 팟타이 줄이 서 있다. 쌀국수면을 강한 불에 빠르게 볶고, 새우, 두부, 부추, 숙주, 라임, 땅콩가